# MRE — Phase 3 & 4: Closed-Loop Pipeline, Benchmarks & Ablation

This notebook demonstrates the complete MRE system:

| Section | Content |
|---------|--------|
| §1 | EvaluationCommission — 4-dimensional scoring |
| §2 | SelectionEngine — Elo ratings & culling |
| §3 | EvolutionEngine — crossover, mutation, replenishment |
| §4 | MREPipeline — full closed loop on algebra problems |
| §5 | BenchmarkRunner — accuracy across 20 synthetic problems |
| §6 | AblationStudy — component contribution analysis |
| §7 | Convergence curve & visualisation |


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))

from mre.agents.dna import AgentDNA
from mre.agents.state import ReasoningState
from mre.operators.pipeline import OperatorPipeline
from mre.evaluation.commission import EvaluationCommission
from mre.evolution.selection import SelectionEngine
from mre.evolution.engine import EvolutionEngine
from mre.pipeline import MREPipeline
from mre.benchmarks import SyntheticBenchmark, BenchmarkRunner
from mre.ablation import AblationStudy

print('✓ All Phase 3+4 modules imported')

## §1 — EvaluationCommission


In [ ]:
# Solve a real problem and score it
pipe = OperatorPipeline.from_names(['SymbolicSimplify','EquationSolve','SelfCritique'])
state = ReasoningState.from_problem('Solve x**2-5*x+6=0')
state = state.evolve(context={'equation':'x**2-5*x+6=0'})
result = pipe.run(state)

commission = EvaluationCommission()
verdict = commission.evaluate(result, expected_answer='2')
print(verdict.report())

In [ ]:
# Compare a correct vs incorrect result
from mre.agents.state import StepRecord
from mre.operators.pipeline import PipelineResult

def make_mock(solved, answer, steps=2):
    s = ReasoningState.from_problem('p')
    if solved: s = s.set_answer(answer)
    s = s.evolve(history=[StepRecord(f'Op{i}',f'in{i}',f'out{i}',True) for i in range(steps)])
    return PipelineResult(s, [], 0.1, solved, answer if solved else None, [])

for label, r, exp in [
    ('Correct  ', make_mock(True,  '[2, 3]', 2), '2'),
    ('Wrong    ', make_mock(True,  '[99]',   2), '2'),
    ('No answer', make_mock(False, None,     3), '2'),
]:
    v = commission.evaluate(r, exp)
    print(f'{label} | score={v.weighted_score:.4f} | '
          f'correct={v.correctness:.2f} logic={v.logic:.2f} '
          f'critique={v.critique:.2f} concise={v.conciseness:.2f}')

## §2 — SelectionEngine (Elo)


In [ ]:
import random

se = SelectionEngine(cull_fraction=0.25)
population = [
    AgentDNA(reasoning_gene=['SymbolicSimplify','EquationSolve','SelfCritique'],   prompt_gene='rigorous'),
    AgentDNA(reasoning_gene=['EquationSolve','SelfCritique','RepairChain'],         prompt_gene='concise'),
    AgentDNA(reasoning_gene=['SymbolicSimplify','EquationSolve'],                   prompt_gene='creative'),
    AgentDNA(reasoning_gene=['DeductiveStep','EquationSolve','SelfCritique'],       prompt_gene='adversarial'),
]

# Simulate two rounds
for round_scores in [[0.92, 0.75, 0.60, 0.30], [0.88, 0.80, 0.55, 0.35]]:
    se.update(population, round_scores)

print(se.leaderboard(population))
survivors, culled = se.select(population)
print(f'\nSurvivors: {len(survivors)} | Culled: {[d.agent_id for d in culled]}')

## §3 — EvolutionEngine


In [ ]:
rng = random.Random(42)
ee  = EvolutionEngine(selector=se, target_population=4, mutation_rate=0.3, rng=rng)

print('Before evolution:')
for d in population:
    print(f'  {d.agent_id}  ops={d.reasoning_gene}  prompt={d.prompt_gene}')

new_pop = ee.evolve(population, [0.92, 0.75, 0.60, 0.30])
print()
print(ee.last_summary.report())
print()
print('After evolution:')
for d in new_pop:
    parent_str = f'← {d.parent_ids}' if d.parent_ids else '← seed'
    print(f'  {d.agent_id}  gen={d.generation}  ops={d.reasoning_gene}  {parent_str}')

## §4 — MREPipeline: Closed Loop


In [ ]:
problems = [
    {'text': 'Solve x**2-5*x+6=0', 'context': {'equation':'x**2-5*x+6=0'}, 'answer': '2'},
    {'text': 'Solve 3*x+9=0',       'context': {'equation':'3*x+9=0'},       'answer': '-3'},
    {'text': 'Simplify (x**2-1)/(x-1)', 'context': {}, 'answer': 'x + 1'},
]

pipeline = MREPipeline(
    population_size=4,
    generations=4,
    target_score=0.88,
    mutation_rate=0.3,
    cull_fraction=0.25,
)

history = pipeline.run(problems)
print()
pipeline.print_history(history)

In [ ]:
# Convergence curve
import matplotlib.pyplot as plt

curve = pipeline.convergence_curve(history)
fig, ax = plt.subplots(figsize=(9,4))
fig.patch.set_facecolor('#0f0f1a')
ax.set_facecolor('#1a1a2e')
ax.plot(range(1, len(curve)+1), curve, '-o', color='#6BCB77', linewidth=2.5, markersize=8)
ax.axhline(0.88, color='#FFD93D', linestyle='--', alpha=0.7, label='target=0.88')
ax.fill_between(range(1, len(curve)+1), curve, alpha=0.15, color='#6BCB77')
ax.set_xlabel('Generation', color='white'); ax.set_ylabel('Best Commission Score', color='white')
ax.set_title('MRE Convergence Curve', color='white', fontweight='bold')
ax.tick_params(colors='white'); ax.legend(labelcolor='white', facecolor='#1a1a2e')
for sp in ax.spines.values(): sp.set_edgecolor('#444466')
plt.tight_layout(); plt.show()

## §5 — BenchmarkRunner


In [ ]:
bench  = SyntheticBenchmark(n=20, seed=42)
runner = BenchmarkRunner(population_size=3, generations=2, target_score=0.99)
result = runner.run(bench.problems())

print(result.summary())
print()
print('Per-problem detail:')
for pp in result.per_problem:
    mark = '✓' if pp['correct'] else '✗'
    print(f'  {mark} [{pp["difficulty"]:6}] {pp["text"][:45]:45s} '
          f'got={str(pp["got"])[:20]}')

In [ ]:
# Dashboard plot
runner.plot(result)

## §6 — Ablation Study


In [ ]:
study = AblationStudy(
    n_problems=10,
    generations=2,
    population_size=3,
    conditions=['full', 'no_evolution', 'single_agent', 'no_repair', 'no_simplify'],
)
ablation_results = study.run()
study.print_table(ablation_results)

In [ ]:
study.plot(ablation_results)

## §7 — Full System Summary


In [ ]:
print('╔══════════════════════════════════════════════════════════════════════╗')
print('║  MRE System — Phase 1-4 Complete                                    ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print('║  Phase 1: NRO knowledge graph (ProofWiki + SyntheticKG)             ║')
print('║  Phase 2: AgentDNA (5 genes) + 6 reasoning operators                ║')
print('║  Phase 3: EvaluationCommission + Elo selection + EvolutionEngine    ║')
print('║  Phase 4: BenchmarkRunner (MATH/miniF2F) + AblationStudy            ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f'║  Benchmark accuracy : {result.accuracy:.1%} on {len(result.problems)} synthetic problems      ║')
print(f'║  Evolution loop     : {len(history)} generations, best={max(curve):.4f}                  ║')
print('╚══════════════════════════════════════════════════════════════════════╝')